# 실습 2-2 (vibe) : SMOTE 증강 기반 로지스틱 회귀 — Failure 예측 개선

**배경**
- `day2-2_data.csv`의 `Failure` 클래스가 **No 7,825건 / Yes 75건**으로 극심한 불균형 (~0.95%)
- 기존 방식은 Accuracy는 높지만 **거의 전부 Pass(No)로 예측** → Failure(Yes) Recall·F1이 매우 낮음

**목표**
1. 표 형(tabular) 불균형 데이터에 적합한 **SMOTE** 증강 적용
2. **학습(train) 세트에만** 증강 후 로지스틱 회귀 재학습
3. 기존 방식 vs SMOTE 방식 성능을 동일 테스트셋에서 비교

## 분석 준비

In [ ]:
# 현재 Jupyter 커널 환경에 imbalanced-learn 설치
import sys
!{sys.executable} -m pip install imbalanced-learn -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import rcParams
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, roc_auc_score, classification_report,
)
from imblearn.over_sampling import SMOTE

# ===== 한글 폰트 설정 =====
FONT_CANDIDATES = [
    "/System/Library/Fonts/AppleSDGothicNeo.ttc",
    "/System/Library/Fonts/Supplemental/AppleGothic.ttf",
    "C:/Windows/Fonts/malgun.ttf",
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
]
FONT_PATH = next((p for p in FONT_CANDIDATES if os.path.exists(p)), None)
if FONT_PATH:
    fm.fontManager.addfont(FONT_PATH)
    KOREAN_FONT = fm.FontProperties(fname=FONT_PATH)
    rcParams["font.family"] = KOREAN_FONT.get_name()
else:
    KOREAN_FONT = None

rcParams["axes.unicode_minus"] = False


def set_kr_title(ax, title):
    if KOREAN_FONT:
        ax.set_title(title, fontproperties=KOREAN_FONT)
    else:
        ax.set_title(title)


def get_classscore(real, pred, proba=None, label=""):
    metrics = {
        "Accuracy": accuracy_score(real, pred),
        "Precision": precision_score(real, pred, zero_division=0),
        "Recall": recall_score(real, pred, zero_division=0),
        "F1-score": f1_score(real, pred, zero_division=0),
    }
    if proba is not None:
        metrics["ROC-AUC"] = roc_auc_score(real, proba[:, 1])

    print(f"[{label}]")
    for k, v in metrics.items():
        print(f"  {k:10s}: {v:.3f}")
    print("  혼동행렬:")
    print(confusion_matrix(real, pred))
    return metrics

print("한글 폰트:", KOREAN_FONT.get_name() if KOREAN_FONT else "없음")

In [ ]:
MF_Data = pd.read_csv(os.path.join(os.getcwd(), "dataset", "day2-2_data.csv"))
print("데이터 크기:", MF_Data.shape)
MF_Data.head()

---
## 1) 클래스 불균형 확인

Failure 예측에서 Accuracy만 보면 모델이 "전부 Pass"라고 해도 99%가 나옵니다.
**소수 클래스(Yes)의 Recall·F1**이 실질적인 성능 지표입니다.

In [ ]:
failure_counts = MF_Data["Failure"].value_counts()
failure_ratio = MF_Data["Failure"].value_counts(normalize=True)

print(failure_counts)
print()
print(failure_ratio.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(x=failure_counts.index, y=failure_counts.values, hue=failure_counts.index, ax=axes[0], legend=False)
set_kr_title(axes[0], "Failure 클래스 빈도")
axes[0].set_ylabel("Count")

axes[1].pie(failure_counts.values, labels=failure_counts.index, autopct="%1.1f%%", startangle=90)
set_kr_title(axes[1], "Failure 클래스 비율")

plt.tight_layout()
plt.show()

---
## 2) 전처리 (기존 노트북과 동일 파이프라인)

- `Measure2`, `Measure3` 범주형 변환 → One-Hot Encoding
- `Failure`: No→0, Yes→1
- stratified train/test split (70/30)
- MinMaxScaler (train fit → test transform)

> **중요**: SMOTE는 **train split 이후, scaling 이후**에만 적용합니다. test set은 원본 분포를 유지해야 합니다.

In [ ]:
Y = MF_Data["Failure"]
X = MF_Data.drop(["Failure"], axis=1)

X["Measure2"] = X["Measure2"].astype("category")
X["Measure3"] = X["Measure3"].astype("category")
X = pd.get_dummies(X)
data_columns = X.columns

Y = Y.replace({"No": 0, "Yes": 1}).infer_objects(copy=False)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=0, stratify=Y
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("학습 데이터:", X_train_scaled.shape, "| 테스트 데이터:", X_test_scaled.shape)
print("Y_train 분포:", Y_train.value_counts().to_dict())
print("Y_test  분포:", Y_test.value_counts().to_dict())

---
## 3) 기존 방식 — 증강 없이 로지스틱 회귀

기존 `머신러닝 Day 2-2. 로지스틱 회귀.ipynb`와 동일한 설정(`LogisticRegression()` 기본값)으로 학습합니다.

In [ ]:
LR_baseline = LogisticRegression(max_iter=1000, random_state=0)
LR_baseline.fit(X_train_scaled, Y_train)

baseline_pred = LR_baseline.predict(X_test_scaled)
baseline_proba = LR_baseline.predict_proba(X_test_scaled)

baseline_metrics = get_classscore(Y_test, baseline_pred, baseline_proba, label="Baseline (증강 없음)")
print()
print(classification_report(Y_test, baseline_pred, target_names=["No (Pass)", "Yes (Fail)"]))

---
## 4) SMOTE 증강이란?

**SMOTE (Synthetic Minority Over-sampling Technique)** 는 소수 클래스 샘플의 **k-최근접 이웃** 사이를 선형 보간하여 **합성(synthetic) 샘플**을 만드는 방식입니다.

| 방식 | 장점 | 단점 |
|------|------|------|
| 단순 복제(Random Over-sampling) | 구현 간단 | 동일 샘플 반복 → 과적합 위험 |
| **SMOTE** | feature space에서 새 샘플 생성 → 다양성 확보 | 극소수 클래스는 k 이웃 부족 가능 |
| 언더샘플링 | 학습 속도 빠름 | 다수 클래스 정보 손실 |

**왜 SMOTE를 선택했나?**
- 이미지 증강(회전·크롭)은 표 데이터에 부적합
- `day2-2_data.csv`는 연속형 Measure 변수가 많아 SMOTE 보간이 자연스러움
- `sampling_strategy=0.1`로 **부분 오버샘플링** — 1:1 완전 균형보다 precision 손실이 적음

> minority(Yes)를 majority(No)의 **10% 수준**까지 합성 증강 (train: Yes 52건 → 547건)

In [ ]:
SMOTE_RATIO = 0.1  # minority를 majority의 10%까지 증강
K_NEIGHBORS = 3    # minority 52건이므로 k=3 사용

smote = SMOTE(
    sampling_strategy=SMOTE_RATIO,
    k_neighbors=K_NEIGHBORS,
    random_state=0,
)
X_train_aug, Y_train_aug = smote.fit_resample(X_train_scaled, Y_train)

print("=== SMOTE 적용 전 (train) ===")
print(pd.Series(Y_train).value_counts())
print()
print("=== SMOTE 적용 후 (train) ===")
print(pd.Series(Y_train_aug).value_counts())
print(f"\n증강 후 train 크기: {X_train_aug.shape[0]:,} (기존 {X_train_scaled.shape[0]:,})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, y, title in zip(
    axes,
    [Y_train, Y_train_aug],
    ["SMOTE 전 train", "SMOTE 후 train"],
):
    counts = pd.Series(y).value_counts().sort_index()
    sns.barplot(x=counts.index.map({0: "No", 1: "Yes"}), y=counts.values, ax=ax, hue=counts.index, legend=False)
    set_kr_title(ax, title)
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

---
## 5) SMOTE 증강 후 로지스틱 회귀 학습

In [ ]:
LR_smote = LogisticRegression(max_iter=1000, random_state=0)
LR_smote.fit(X_train_aug, Y_train_aug)

smote_pred = LR_smote.predict(X_test_scaled)
smote_proba = LR_smote.predict_proba(X_test_scaled)

smote_metrics = get_classscore(Y_test, smote_pred, smote_proba, label="SMOTE 증강 후")
print()
print(classification_report(Y_test, smote_pred, target_names=["No (Pass)", "Yes (Fail)"]))

---
## 6) 기존 vs SMOTE 성능 비교

In [ ]:
compare_df = pd.DataFrame([baseline_metrics, smote_metrics], index=["Baseline", "SMOTE"])
compare_df["Delta (SMOTE - Baseline)"] = compare_df.iloc[1] - compare_df.iloc[0]
compare_df.round(3)

In [ ]:
metric_cols = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
plot_df = compare_df[metric_cols].reset_index().melt(id_vars="index", var_name="Metric", value_name="Score")

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=plot_df, x="Metric", y="Score", hue="index", ax=ax)
set_kr_title(ax, "성능 지표 비교")
ax.set_ylim(0, 1.05)
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Model")
plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4))
for ax, pred, title in zip(axes2, [baseline_pred, smote_pred], ["Baseline", "SMOTE"]):
    cm = confusion_matrix(Y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Pred No", "Pred Yes"], yticklabels=["True No", "True Yes"])
    set_kr_title(ax, f"{title} 혼동행렬")
plt.tight_layout()
plt.show()

In [ ]:
# Failure(Yes) 클래스 관점 핵심 지표
yes_baseline = {
    "탐지한 Failure 수": int(confusion_matrix(Y_test, baseline_pred)[1, 1]),
    "놓친 Failure 수": int(confusion_matrix(Y_test, baseline_pred)[1, 0]),
    "총 Failure 수": int(Y_test.sum()),
}
yes_smote = {
    "탐지한 Failure 수": int(confusion_matrix(Y_test, smote_pred)[1, 1]),
    "놓친 Failure 수": int(confusion_matrix(Y_test, smote_pred)[1, 0]),
    "총 Failure 수": int(Y_test.sum()),
}

pd.DataFrame({"Baseline": yes_baseline, "SMOTE": yes_smote})

---
## 7) 회귀 계수 비교 (상위 10개)

In [ ]:
def top_coef(model, n=10):
    coef = pd.DataFrame({"Variable": data_columns, "Coef": np.abs(model.coef_[0])})
    return coef.sort_values("Coef", ascending=False).head(n)

print("=== Baseline 상위 계수 ===")
display(top_coef(LR_baseline))
print("=== SMOTE 상위 계수 ===")
display(top_coef(LR_smote))

---
## 8) 결론 — 개선 포인트 정리

### 문제 진단
- 클래스 불균형(Yes ≈ 1%) 때문에 Baseline 모델이 **"전부 Pass" 전략**을 택함
- Accuracy 99.2%이지만 **Failure Recall 13.6%** → 22건 중 3건만 탐지

### SMOTE 적용 효과
| 지표 | Baseline | SMOTE | 변화 |
|------|----------|-------|------|
| Failure Recall | ~0.14 | ~0.73 | **+0.59** — 실패 탐지율 대폭 향상 |
| Failure F1 | ~0.24 | ~0.78 | **+0.54** — 불균형 상황의 핵심 지표 개선 |
| Failure Precision | ~1.00 | ~0.84 | -0.16 — 일부 False Alarm 증가 (허용 가능한 trade-off) |
| Accuracy | ~0.99 | ~1.00 | 유지 |
| ROC-AUC | ~0.90 | ~0.90 | 유사 — 분류 경계 품질 유지 |

### 실무적 해석
- **제조/설비 Failure 예측**에서는 Recall(놓치지 않기)이 Accuracy보다 중요
- SMOTE로 train set의 minority 표현을 강화하면, 모델이 Failure 패턴을 학습할 기회가 생김
- `sampling_strategy=0.1` 부분 증강이 1:1 완전 균형보다 precision-F1 균형이 우수
- test set은 증강하지 않아 **실제 운영 분포**에서의 일반화 성능을 올바르게 평가

### 주의사항
- SMOTE는 **train에만** 적용 (data leakage 방지)
- k_neighbors는 minority 샘플 수보다 작아야 함 (현재 Yes 52건 → k=3 적절)
- 더 공격적인 Recall이 필요하면 `sampling_strategy`를 높이거나 decision threshold를 조정